In [0]:
%run ./lib/00_common

In [0]:
%run ./lib/01_exceptions

In [0]:
%run ./lib/02_delta_writer

In [0]:
%run ./lib/03_logger_utils

In [0]:
%run ./lib/04_quality_pandas

In [0]:
%run ./lib/06_alerting

In [0]:
import pandas as pd

run_id = get_text_widget("run_id", "").strip()
run_date = get_text_widget("run_date", "").strip()

if not run_id:
    raise ValueError("run_id es obligatorio")
if not run_date:
    raise ValueError("run_date es obligatorio")

alert_email_to = get_text_widget("alert_email_to", "")
gmail_smtp_user = get_text_widget("gmail_smtp_user", "")
gmail_app_password = get_text_widget("gmail_app_password", "")

In [0]:
# =========================
# 1. Orders -> Silver
# =========================
try:
    stage = "silver_orders"
    dataset = "orders"

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="STARTED",
        message="Inicia Silver orders con pandas-first"
    )

    current_orders_file = assert_file_exists(bronze_orders_snapshot_file(run_date))
    bronze_orders_current_pdf = pd.read_csv(current_orders_file)

    current_total_rows = len(bronze_orders_current_pdf)
    if current_total_rows == 0:
        raise DataQualityError(f"No hay datos en el snapshot Bronze del día {run_date}")

    valid_current_pdf, invalid_current_pdf = split_valid_invalid_orders_pdf(bronze_orders_current_pdf)
    current_reject_rate = compute_reject_rate(current_total_rows, len(invalid_current_pdf))

    if current_reject_rate > REJECT_THRESHOLD:
        raise DataQualityError(
            f"Reject rate alto en run_date={run_date}: {current_reject_rate:.2%}. "
            f"Límite={REJECT_THRESHOLD:.2%}"
        )

    bronze_orders_all_pdf = read_all_csv(BRONZE_WORK_ORDERS_ROOT)
    if bronze_orders_all_pdf.empty:
        raise DataQualityError("No hay snapshots Bronze históricos para orders")

    valid_all_pdf, invalid_all_pdf = split_valid_invalid_orders_pdf(bronze_orders_all_pdf)

    silver_orders_pdf = valid_all_pdf[[
        "order_id", "customer_id", "order_date", "amount", "status",
        "_ingestion_ts", "_source_file", "_run_id", "_run_date"
    ]].copy()

    write_csv(silver_orders_pdf, silver_orders_current_file())

    write_reject_orders_delta(invalid_all_pdf, mode="overwrite")
    write_silver_orders_delta(silver_orders_pdf, mode="overwrite")

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="OK",
        message="Silver orders completado",
        rows_ok=len(silver_orders_pdf),
        rows_reject=len(invalid_all_pdf),
        extra={
            "current_run_rows": current_total_rows,
            "current_run_rejects": len(invalid_current_pdf),
            "current_run_reject_rate": round(current_reject_rate, 4),
            "historical_valid_rows": len(valid_all_pdf),
            "historical_invalid_rows": len(invalid_all_pdf)
        }
    )

except Exception as e:
    error_code = "DQ_001" if isinstance(e, DataQualityError) else "SYS_001"
    notify_failure(
        spark=spark,
        run_id=run_id,
        stage="silver_orders",
        dataset="orders",
        error_message=str(e),
        error_code=error_code,
        smtp_user=gmail_smtp_user,
        smtp_app_password=gmail_app_password,
        to_email=alert_email_to
    )
    raise

In [0]:

# =========================
# 2. Customers -> Silver
# =========================
try:
    stage = "silver_customers"
    dataset = "customers"

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="STARTED",
        message="Inicia Silver customers con pandas-first"
    )

    bronze_customers_all_pdf = read_all_csv(BRONZE_WORK_CUSTOMERS_ROOT)
    if bronze_customers_all_pdf.empty:
        raise DataQualityError("No hay snapshots Bronze históricos para customers")

    silver_customers_pdf = dedupe_customers_pdf(bronze_customers_all_pdf)

    write_csv(silver_customers_pdf, silver_customers_current_file())
    write_silver_customers_delta(silver_customers_pdf, mode="overwrite")

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="OK",
        message="Silver customers completado",
        rows_ok=len(silver_customers_pdf),
        rows_reject=len(bronze_customers_all_pdf) - len(silver_customers_pdf),
        extra={"historical_source_rows": len(bronze_customers_all_pdf)}
    )

except Exception as e:
    error_code = "DQ_002" if isinstance(e, DataQualityError) else "SYS_002"
    notify_failure(
        spark=spark,
        run_id=run_id,
        stage="silver_customers",
        dataset="customers",
        error_message=str(e),
        error_code=error_code,
        smtp_user=gmail_smtp_user,
        smtp_app_password=gmail_app_password,
        to_email=alert_email_to
    )
    raise

print(f"Silver pandas-first OK. run_id={run_id}")